# Deux zones et un lien : flux, duale de la capacité d'échange et rente de congestion

*Deux zones reliées par une ligne de 1 600 MW, du PV abondant au sud et une turbine à gaz chère au nord. Lire les flux, le prix de chaque zone et la duale de la capacité du lien, puis vérifier qu'elle vaut l'écart de prix moins le hurdle cost aux heures saturées et l'économie d'un MW de plus. Élargir ensuite la ligne et voir l'écart de prix fondre.*

Ce notebook est tiré du texte [en ligne](https://www.robingirard.eu/assets/parcours-es/latest/textes/act-transport-deux-zones.html) du parcours ES ; exécutez les cellules dans l'ordre (Maj + Entrée). Les commentaires des cellules donnent les valeurs attendues.

**La cellule d'installation.** En local, placez ce notebook dans le dépôt `pommes-case-studies` (à sa racine ou dans un de ses dossiers) et ouvrez-le avec le noyau du venv du dépôt (`.venv`) : la cellule n'installe rien, elle trouve le dépôt et affiche les versions des paquets.

Sur Google Colab (Fichier › Importer un notebook), la cellule clone le dépôt et installe les paquets : 2 à 3 minutes, puis elle s'arrête volontairement sur `RedemarrezLaSession`. Faites Exécution › Redémarrer la session, relancez cette cellule, et seulement elle, puis continuez dans l'ordre.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

VERSIONS = {"pommes": "0.11.0", "pommes-craft": "0.6.0", "linopy": "0.10.1", "highspy": "1.15.1"}


def versions_du_depot() -> bool:
    """Vrai si les paquets installés sont ceux de requirements.txt (les valeurs attendues en dépendent)."""
    try:
        return all(version(nom) == v for nom, v in VERSIONS.items())
    except PackageNotFoundError:
        return False


class RedemarrezLaSession(Exception):
    """Colab : les paquets sont installés, mais la session garde en mémoire les anciens numpy et pandas."""


if "google.colab" in sys.modules:                     # sur Colab : cloner, installer, redémarrer une fois
    CLONE = Path("/content/pommes-case-studies")
    if not CLONE.exists():
        !git clone https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies {CLONE}
    %cd {CLONE}
    if not versions_du_depot():
        !python --version
        !pip install -q -r requirements.txt
        raise RedemarrezLaSession("Installation finie. Exécution › Redémarrer la session, "
                                  "puis relancez cette cellule (et seulement elle).")

RACINE = next((d for d in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "pommes-case-studies")
               if (d / "pommes_cases").is_dir()), Path.cwd())  # la racine du dépôt des cas
if not (RACINE / "pommes_cases").is_dir():
    raise FileNotFoundError(f"pas de pommes_cases/ sous {RACINE} : placez ce notebook dans le dépôt "
                            "pommes-case-studies (à sa racine ou dans un de ses dossiers)")
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
print("dépôt :", RACINE.name, "· Python", sys.version.split()[0])
print(" · ".join(f"{nom} {version(nom)}" for nom in VERSIONS))
if not versions_du_depot():
    print("Attention : ce ne sont pas les versions du dépôt", VERSIONS, ": les valeurs attendues peuvent bouger.")

## Objectif

Relier deux zones par une ligne bornée, lire leurs prix et la duale de la capacité du lien, puis vérifier par deux résolutions qu'elle vaut un MW de plus.

Prérequis : le dépôt des cas installé [Installer et vérifier en dix minutes : venv du dépôt des cas, test de contrôle, dépannage](https://www.robingirard.eu/assets/parcours-es/latest/textes/installer.html) ; le tutoriel d'une zone [Le premier modèle sur une page : les huit étapes du notebook et leurs valeurs attendues](https://www.robingirard.eu/assets/parcours-es/latest/textes/premier-modele.html) ; les duales de l'équilibre offre-demande et d'une capacité (interconnexion saturée), § 5 de [Le prix est une duale : trois centrales à la main, en LP, puis dans pommes_craft](https://www.robingirard.eu/assets/parcours-es/latest/textes/prix-dual.html). Neuf résolutions de deux à trois secondes, construction comprise. Valeurs du 16/11/2026, revérifiées le 17/11/2026 avec pommes 0.11.0, pommes_craft 0.6.0 et linopy 0.10.1.

## 1. Le modèle

Sud : demande plate de 1 500 MW, 5 000 MW de PV au facteur de charge horaire de l'Espagne (jeu `pecd2026`, 2030, météo par défaut, moyenne 0,181), 5 000 MW de cycle combiné à 60 €/MWh. Nord : demande de 1 000 MW à 3 h à 3 000 MW à 15 h, 3 000 MW de turbine à gaz à 120 €/MWh. Délestage à 3 000 €/MWh des deux côtés. Capacités imposées (`power_capacity_investment_min = max`), sans coût d'investissement, comme dans `pommes_cases/systeme_fr.py`.

Le lien : un `Link` de Sud vers Nord et une `TransportTechnology` qui porte la capacité, imposée de même, avec un `hurdle_costs` de 1 €/MWh transporté. Un `Link` est orienté (`operation_transport_power` positive ou nulle, de `area_from` vers `area_to`) ; le Nord étant toujours plus cher, un seul suffit.

In [ ]:
import sys, logging, tempfile
from pathlib import Path
import numpy as np, polars as pl
from pommes_craft import (EnergyModel, Area, Link, TransportTechnology, ConversionTechnology, Demand,
                          EconomicHypothesis, TimeStepManager, LoadShedding)
from pommes_cases.solve import build_lp, solve_with_extra_constraints
from pommes_cases.meteo import load_meteo, availability
from pommes.utils import dual_scale
logging.getLogger().setLevel(logging.ERROR)

H, Y = 8760, 2030
h = np.arange(H)
pv_dispo = availability(load_meteo("ES", Y), "pv")                      # facteur de charge PV, Espagne, WS003
demande_nord = 2000 + 1000 * np.sin(2 * np.pi * (h % 24 - 9) / 24)      # 1 000 MW à 3 h, 3 000 MW à 15 h
frame = lambda v, col: pl.DataFrame({"hour": h, "year_op": np.full(H, Y), col: v}).with_columns(pl.col("hour", "year_op").cast(pl.Int64))

def fixe(nom, mw, **kw):   # capacité existante, imposée, sans coût d'investissement
    return ConversionTechnology(name=nom, factor={"electricity": 1.0}, life_span=1000, invest_cost=0.0,
                                power_capacity_investment_min=mw, power_capacity_investment_max=mw, **kw)

def modele(capacite_lien, hurdle=1.0):
    m = EnergyModel(name="deux_zones", folder=Path(tempfile.mkdtemp()), hours=list(h), year_ops=[Y],
                    year_invs=[Y], year_decs=[Y + 1000], modes=[], resources=["electricity"])
    with m.context():
        EconomicHypothesis("eco", discount_rate=0.0, year_ref=Y, planning_step=1)
        TimeStepManager("pas", time_step_duration=1.0, operation_year_duration=float(H))
        sud, nord = Area("Sud"), Area("Nord")
        for zone, dem in ((sud, np.full(H, 1500.0)), (nord, demande_nord)):
            zone.add_component(Demand(name="demande", resource="electricity", demand=frame(dem, "demand")))
            zone.add_component(LoadShedding(name="delestage", resource="electricity", cost=3000.0))
        sud.add_component(fixe("pv", 5000.0, availability=pv_dispo))
        sud.add_component(fixe("ccg", 5000.0, variable_cost=60.0))
        nord.add_component(fixe("tac", 3000.0, variable_cost=120.0))
        lien = Link(name="sud_nord", area_from=sud, area_to=nord)          # un Link = un sens
        lien.add_transport_technology(TransportTechnology(
            name="ligne", resource="electricity", life_span=1000, invest_cost=0.0, hurdle_costs=hurdle,
            power_capacity_investment_min=capacite_lien, power_capacity_investment_max=capacite_lien))
    return m

## 2. Résoudre et lire flux et prix

Le flux, en MW : la variable `operation_transport_power` (dimensions `link`, `transport_tech`, `resource`, `hour`, `year_op`). Le prix de chaque zone : la duale de `operation_adequacy_constraint` (dimensions `area`, `hour`, `resource`, `year_op` ; 8 760 valeurs par zone), divisée par `dual_scale(params)`, qui vaut 1 ici (actualisation nulle, 8 760 heures). La duale de la capacité : celle de `operation_transport_power_max_constraint`, contrainte ≤ que linopy rend négative en minimisation, d'où le signe moins. On lit `lp.solution` : avant pommes 0.11.0, `get_capacities` et les résumés de pommes_craft comptaient un lien comme import de la zone exportatrice (CHANGELOG, #157).

In [ ]:
def lire(capacite_lien, hurdle=1.0):
    m = modele(capacite_lien, hurdle)
    _, params = build_lp(m)
    lp = solve_with_extra_constraints(m)
    s = float(dual_scale(params).squeeze())                                # 1.0 ici
    flux = lp.solution["operation_transport_power"].squeeze().values      # MW, de Sud vers Nord
    prix = lp.dual["operation_adequacy_constraint"].squeeze() / s         # €/MWh, par zone
    p_sud, p_nord = prix.sel(area="Sud").values, prix.sel(area="Nord").values
    mu = -lp.dual["operation_transport_power_max_constraint"].squeeze().values / s   # €/MW, duale de la capacité
    sature = flux > capacite_lien - 1e-6
    return dict(cout=float(lp.objective.value), flux=flux, p_sud=p_sud, p_nord=p_nord, mu=mu, sature=sature,
                rente=float((flux * (p_nord - p_sud)).sum()), scale=s)

r = lire(1600.0)
print(f"coût total {r['cout']:,.0f} € ; énergie transportée {r['flux'].sum()/1e6:.2f} TWh ; heures saturées {r['sature'].sum()}")
print("prix Sud :", np.unique(np.round(r['p_sud'], 1) + 0.0), "; prix Nord :", np.unique(np.round(r['p_nord'], 1)))   # + 0.0 : -0. devient 0.
print(f"prix moyen Sud {r['p_sud'].mean():.2f}, Nord {r['p_nord'].mean():.2f} €/MWh ; rente {r['rente']/1e6:.2f} M€")
# coût total 1,671,805,728 € ; énergie transportée 12.75 TWh ; heures saturées 5475
# prix Sud : [ 0. 60.] ; prix Nord : [ 61. 120.]
# prix moyen Sud 55.93, Nord 97.88 €/MWh ; rente 586.61 M€

Flux du Sud au Nord toutes les heures. Prix du Sud : 0 quand le PV est écrêté, 60 quand le cycle combiné est marginal. Prix du Nord : 120 quand la turbine tourne, 61 sinon, soit le prix du Sud plus le hurdle cost : lien non saturé, le MWh de plus au Nord vient du Sud. La rente de congestion, Σ flux × (prix Nord − prix Sud), vaut 586,6 M€ pour 1 600 MW.

## 3. La duale de la capacité

In [ ]:
sat = r["sature"]
print(f"Σ mu {r['mu'].sum():,.0f} €/MW ; Σ (Nord - Sud - hurdle) aux heures saturées {(r['p_nord'] - r['p_sud'] - 1.0)[sat].sum():,.0f}")
print(f"mu max hors saturation {r['mu'][~sat].max():.3f} ; écart Nord - Sud hors saturation {np.unique((r['p_nord'] - r['p_sud'])[~sat].round(2))}")
# Σ mu 358,665 €/MW ; Σ (Nord - Sud - hurdle) aux heures saturées 358,665
# mu max hors saturation 0.000 ; écart Nord - Sud hors saturation [1.]

Aux 5 475 heures saturées, la duale vaut l'écart de prix moins le hurdle cost : 4 881 heures à 60 − 1, 594 à 120 − 1. Hors saturation, elle est nulle. Sur l'année, 358 665 €/MW : ce que rapporterait un MW de ligne en plus. La rente s'en déduit : 1 600 × (358 665 + 1 × 5 475) plus le hurdle sur le flux hors saturation, soit 586,6 M€.

## 4. Vérifier par une seconde résolution

In [ ]:
r1 = lire(1601.0)
print(f"baisse du coût pour 1 MW de plus {r['cout'] - r1['cout']:,.0f} € ; Σ mu {r['mu'].sum():,.0f} €")
# baisse du coût pour 1 MW de plus 358,485 € ; Σ mu 358,665 €

Écart de 0,05 %, parce que le MW ajouté change les heures saturées à la marge : la duale est bien la dérivée du coût total par rapport à la capacité.

## 5. Élargir le lien

In [ ]:
for cap in [800.0, 1200.0, 1600.0, 2400.0, 2800.0, 3200.0]:
    x = lire(cap)
    print(f"{cap:5.0f} MW  h_sat {x['sature'].sum():4d}  Sud {x['p_sud'].mean():5.1f}  Nord {x['p_nord'].mean():5.1f}"
          f"  écart {(x['p_nord'] - x['p_sud']).mean():5.1f} €/MWh  rente {x['rente']/1e6:6.1f} M€  Σ mu {x['mu'].sum()/1e3:6.1f} k€/MW")

```text
  cap    h_sat   Sud    Nord   écart   rente    Σ mu
  800    8760   48.9   120.0   71.1   498.5   614.4
 1200    6935   52.2   107.7   55.5   582.8   477.1
 1600    5475   55.9    97.9   41.9   586.6   358.7
 2400    3285   60.0    83.1   23.1   481.7   193.9
 2800    1825   60.0    73.3   13.3   318.8   107.7
 3200       0   60.0    61.0    1.0    17.5     0.0
```

À chaque élargissement, écart de prix et Σ mu baissent ; le Nord devient moins cher, le Sud plus cher : exporter le PV de midi fait disparaître ses heures à prix nul. À 3 200 MW, plus aucune heure saturée, les prix ne diffèrent que du hurdle cost : une seule zone de prix. La rente monte d'abord, de 498 à 587 M€, puis fond : produit d'une capacité qui grandit par un écart qui rétrécit. Le gestionnaire de réseau vit de la rente, le planificateur regarde Σ mu : pas le même nombre.

En Europe, les gestionnaires de réseau de transport des deux côtés de la frontière se partagent cette rente. L'article 19 du règlement (UE) 2019/943 leur impose d'abord de l'employer à garantir la disponibilité réelle de la capacité allouée et à maintenir ou accroître les capacités d'échange entre zones ; le reste peut venir en déduction des tarifs de réseau. En zonal, comme ici, un prix par zone et la rente concentrée sur le lien ; en nodal, un prix par nœud et la rente répartie sur toutes les lignes saturées.

## Trois questions à réponse chiffrée

1. À 1 600 MW, combien d'heures saturées ont un écart de prix de 120 €/MWh ? (594.)
2. Parmi les six capacités, laquelle maximise la rente, et que vaut alors un MW de plus ? (1 600 MW ; 358,7 k€/MW/an.)
3. Mettez le hurdle cost à 0 : que devient le prix du Nord hors saturation, et Σ mu à 1 600 MW ? (Prévoyez, puis vérifiez : 60 €/MWh au lieu de 61 ; Σ mu passe de 358 665 à 364 140 €/MW, 5 475 € de plus, le hurdle une fois par heure saturée ; les heures saturées ne bougent pas.)

## Vous l'avez acquise quand…

- Vous retrouvez 5 475 heures saturées, les prix 0 et 60 au Sud, 61 et 120 au Nord, à quelles heures ils tombent, et la rente de 586,6 M€.
- Vous expliquez sans calcul pourquoi la duale vaut l'écart de prix moins le hurdle cost, zéro hors saturation.
- Par deux résolutions, vous obtenez une baisse du coût pour 1 MW proche de Σ mu (écart inférieur à 1 %).
- Vous dites en une phrase pourquoi la rente peut monter à l'élargissement alors que Σ mu baisse toujours.

## Pour aller plus loin

- « Two Areas with Link », doc de pommes_craft figée en 0.3.7 : même structure, capacité du lien à investir [Examples de pommes_craft : PV + gaz, deux zones reliées, batterie](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes_craft/examples.html).
- Le cas F, construit : que vaut un GW d'interconnexion France-Espagne ou France-Allemagne, dans un modèle à sept zones (multi-nœuds) et pour la France seule face aux prix de ses voisins [Cas F — Que vaut une interconnexion France-Espagne ou France-Allemagne ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/F_interconnexion).
- Le notebook 2025 à quatre pays, point de départ du cas F, à lire d'un œil critique : sa 14e cellule (index 13 ; `ic_FR_ES.set_attribute('power_capacity_investment_max', 3.6e6)`) renforce FR-ES de 3,6e6 MW, soit 3,6 TW : le résultat « interco ES+ » est celui d'un lien illimité [Notebook 2025 — Valeur des interconnexions en Europe de l'Ouest (ES, FR, IT, DE)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/blob/main/legacy_tp2025/notebooks/Interconnections.ipynb).

## Sources

1. [Dépôt pommes-case-studies (pommes_cases/solve.py, pommes_cases/meteo.py, data/meteo/README.md ; environnement épinglé pommes 0.11.0, pommes_craft 0.6.0, HiGHS)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies)
2. [Code source de pommes (pommes/model/transport.py ; pommes/utils.py, dual_scale) et CHANGELOG (0.11.0 : #133 hurdle costs, #157 imports et exports des liens)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes)
3. [Exemples de pommes_craft, « Two Areas with Link » (doc figée en 0.3.7)](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes_craft/examples.html)
4. [Règlement (UE) 2019/943 sur le marché intérieur de l'électricité, article 19 (recettes de congestion)](https://eur-lex.europa.eu/eli/reg/2019/943/oj)
5. [Même article 19, version anglaise consolidée lisible sans script (legislation.gov.uk)](https://www.legislation.gov.uk/eur/2019/943/article/19)